# Stage 2: Supervised Instruction Fine-Tuning (SFT)

In this stage, we take the model (either base or initialized with the domain knowledge adapters from Stage 1) and fine-tune it on an instruction-response dataset. This teaches the model how to act as a chat assistant and structure its replies.

### Step 1: Install Required Libraries

In [ ]:
# Install Unsloth and standard ML libraries
# Uncomment the lines below to install if running in a GPU cloud env (like Colab)
#!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#!pip install --no-deps "xformers" "trl" peft accelerate bitsandbytes

### Step 2: Load Model and Tokenizer
We load the base model, and can load the Stage 1 adapter if desired, or build SFT directly starting from the base model using the domain datasets.

In [ ]:
from unsloth import FastLanguageModel
import torch
import os

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Option A: Load from base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Option B (Alternative): Load from Stage 1 adapter
# adapters_dir = "adapters" if os.path.exists("data") else "../adapters"
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = os.path.join(adapters_dir, "non_instruction_lora"),
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = load_in_4bit,
# )

### Step 3: Apply LoRA Configuration

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Step 4: Load and Format Instruction Dataset
We load `data/instruction_dataset.jsonl` and format it using the Qwen standard chat template:

`<|im_start|>system\nYou are a helpful IT support assistant.<|im_end|>\n<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{response}<|im_end|>`

In [ ]:
import os
import json
from datasets import Dataset

# Check path compatibility (Colab vs Local)
data_path = "data/instruction_dataset.jsonl" if os.path.exists("data/instruction_dataset.jsonl") else "../data/instruction_dataset.jsonl"

instructions = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            instructions.append(json.loads(line))

chat_template = """<|im_start|>system
You are a professional IT Helpdesk Assistant. Answer the user's technical questions accurately and professionally.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""

def format_prompts(examples):
    texts = []
    for inst, resp in zip(examples["instruction"], examples["response"]):
        text = chat_template.format(instruction=inst, response=resp) + tokenizer.eos_token
        texts.append(text)
    return { "text": texts }

dataset = Dataset.from_dict({
    "instruction": [x["instruction"] for x in instructions],
    "response": [x["response"] for x in instructions]
})
dataset = dataset.map(format_prompts, batched=True)
print(f"Loaded and formatted {len(dataset)} instruction pairs.")

### Step 5: Configure and Run SFTTrainer
We train the model to output technical answers matching the instructions.

In [ ]:
import inspect
from trl import SFTTrainer
from transformers import TrainingArguments

# Try to import SFTConfig for newer TRL versions
try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

if SFTConfig is not None:
    # Newer TRL config style (v0.8.0+)
    sig_config = inspect.signature(SFTConfig.__init__)
    sft_args = {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "warmup_steps": 10,
        "max_steps": 60, # Train longer to learn chat behavior
        "learning_rate": 2e-4,
        "fp16": not torch.cuda.is_bf16_supported(),
        "bf16": torch.cuda.is_bf16_supported(),
        "logging_steps": 1,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 3407,
        "output_dir": "outputs_sft",
        "dataset_text_field": "text",
        "packing": False,
    }
    if "max_seq_length" in sig_config.parameters:
        sft_args["max_seq_length"] = max_seq_length
    else:
        sft_args["max_length"] = max_seq_length
        
    trainer_args = SFTConfig(**sft_args)
else:
    # Older TRL style
    trainer_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60, # Train longer to learn chat behavior
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_sft",
    )

# Prepare trainer arguments
sig_trainer = inspect.signature(SFTTrainer.__init__)
trainer_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "dataset_num_proc": 2,
    "args": trainer_args,
}

if SFTConfig is None:
    # If SFTConfig is None, these must be passed directly to SFTTrainer
    trainer_kwargs["dataset_text_field"] = "text"
    trainer_kwargs["max_seq_length"] = max_seq_length
    trainer_kwargs["packing"] = False

if "processing_class" in sig_trainer.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()

### Step 6: Verify SFT Model Inference
Run inference using our helper prompt to test performance.

In [ ]:
FastLanguageModel.for_inference(model)

eval_prompt = chat_template.format(
    instruction = "Subject: Password reset request\nQuery: I cannot log in, password expired.",
    response = ""
)
inputs = tokenizer([eval_prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

### Step 7: Save SFT Adapters

In [ ]:
import os
# Check path compatibility (Colab/Root vs Local)
adapters_dir = "adapters" if os.path.exists("data") else "../adapters"
save_path = os.path.join(adapters_dir, "sft_lora")

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Stage 2 (SFT) LoRA adapters saved successfully to {save_path}!")